# 📊 Customer Segmentation & Retention Analysis — Executive Summary### Insights Notebook | Telco Customer Churn Dataset---> **Purpose:** This notebook distills findings from five phases of analysis into a clean, narrative-driven summary. It is designed to be read top-to-bottom without executing code — every chart is embedded as a saved figure.---

In [ ]:
import warningswarnings.filterwarnings('ignore')import pandas as pdimport numpy as npimport matplotlib.pyplot as pltimport matplotlib.image as mpimgfrom pathlib import Pathfrom IPython.display import display, Markdown, HTML# PathsFIG_DIR = Path('../reports/figures/')DATA_DIR = Path('../data/processed/')# Styleplt.style.use('seaborn-v0_8-whitegrid')plt.rcParams.update({'figure.dpi': 120, 'font.size': 11})# Load datasetsseg = pd.read_csv(DATA_DIR / 'telco_segmented.csv')clv = pd.read_csv(DATA_DIR / 'telco_clv.csv')test = pd.read_csv(DATA_DIR / 'test_with_predictions.csv')print(f"Datasets loaded: segmented ({seg.shape}), CLV ({clv.shape}), test ({test.shape})")

---## 1. The Business ProblemA telecommunications provider is experiencing significant subscriber attrition. With **7,043 customers** in the dataset, **26.5% have already churned** — but the business lacks the tools to:- **Predict** which customers are at risk *before* they leave- **Understand** the root causes driving churn- **Prioritize** retention spend toward the customers who matter mostThis project answers all three questions using a combination of **unsupervised segmentation**, **supervised classification**, and **lifetime value estimation**.---

## 2. Who Are Our Customers? — Behavioral SegmentationUsing K-Means clustering on five engineered behavioral features, customers naturally group into **three distinct segments** (optimal K selected via silhouette score = 0.42):

In [ ]:
# Segment summary tablesegment_summary = seg.groupby('segment_name').agg(    customers=('Churn_Binary', 'count'),    churn_rate=('Churn_Binary', 'mean'),    avg_tenure=('tenure', 'mean'),    avg_services=('services_count', 'mean'),    avg_risk=('contract_risk_score', 'mean')).reset_index()segment_summary['pct_of_total'] = (segment_summary['customers'] / len(seg) * 100).round(1)segment_summary['churn_rate'] = (segment_summary['churn_rate'] * 100).round(1)segment_summary.columns = ['Segment', 'Customers', 'Churn Rate (%)', 'Avg Tenure (scaled)',                            'Avg Services (scaled)', 'Avg Contract Risk', '% of Total']display(segment_summary[['Segment', 'Customers', '% of Total', 'Churn Rate (%)',                          'Avg Tenure (scaled)', 'Avg Services (scaled)', 'Avg Contract Risk']])

In [ ]:
# Display radar chart — behavioral DNA of each segmentfig, axes = plt.subplots(1, 2, figsize=(16, 6))img1 = mpimg.imread(FIG_DIR / '14_radar_chart.png')axes[0].imshow(img1)axes[0].axis('off')axes[0].set_title('Cluster Behavioral Profiles', fontsize=14, fontweight='bold')img2 = mpimg.imread(FIG_DIR / '15_churn_by_cluster.png')axes[1].imshow(img2)axes[1].axis('off')axes[1].set_title('Churn Rate by Segment', fontsize=14, fontweight='bold')plt.tight_layout()plt.show()

### Key Takeaway — Segmentation> **The At-Risk Churner segment represents 53% of the customer base but generates 84.2% of all churn.** These customers share a common profile: month-to-month contracts, few add-on services, and short tenure. Any retention strategy must prioritize this segment.---

## 3. Why Do Customers Leave? — Churn Drivers (SHAP Analysis)After comparing three models — Logistic Regression (baseline), Random Forest, and XGBoost — the **Random Forest classifier** emerged as the best-performing model:| Metric | Logistic Regression | Random Forest ★ | XGBoost ||---|---|---|---|| AUC-ROC | 0.832 | **0.839** | 0.826 || F1-Score (churn) | 0.601 | **0.625** | 0.603 || Recall (churn) | 0.698 | **0.759** | 0.824 || CV AUC (train) | 0.839 | **0.842** | 0.832 |Using SHAP (SHapley Additive exPlanations), we decomposed every prediction into feature-level contributions:

In [ ]:
# Display SHAP beeswarm — the most insightful single chartfig, axes = plt.subplots(1, 2, figsize=(18, 7))img1 = mpimg.imread(FIG_DIR / '21_shap_beeswarm.png')axes[0].imshow(img1)axes[0].axis('off')axes[0].set_title('SHAP Beeswarm — Feature Impact on Churn Prediction', fontsize=13, fontweight='bold')img2 = mpimg.imread(FIG_DIR / '22_feature_importance.png')axes[1].imshow(img2)axes[1].axis('off')axes[1].set_title('Random Forest Feature Importances', fontsize=13, fontweight='bold')plt.tight_layout()plt.show()

### Top 5 Churn Drivers| Rank | Feature | SHAP Impact | In Plain English ||---|---|---|---|| 1 | **Contract Risk Score** | 0.048 | Month-to-month contracts are the single biggest flight risk || 2 | **Tenure** | 0.045 | Customers who have been around longer are far less likely to leave || 3 | **Contract (Two Year)** | 0.039 | Long-term contracts are powerfully protective || 4 | **Contract (Month-to-Month)** | 0.039 | Confirms: flexibility for the customer = risk for the business || 5 | **Internet Service (Fiber Optic)** | 0.037 | Fiber subscribers churn more — possible quality or pricing issue |> **The pattern is clear:** Contract type dominates churn — 3 of the top 5 drivers are contract-related. Tenure and internet service type round out the picture.---

## 4. How Well Can We Predict Churn?### Threshold Tuning — Optimizing for Business ValueThe default classification threshold (0.50) was tuned to **0.44** to prioritize **recall** — catching more churners is more valuable than reducing false alarms:| Setting | Recall | Precision | F1 ||---|---|---|---|| Default (threshold = 0.50) | 75.9% | 53.1% | 0.625 || **Tuned (threshold = 0.44)** | **81.6%** | 50.7% | 0.625 |> **Result:** The tuned model catches **82 out of every 100 churners** while maintaining a stable F1 score. The 16.9% recall improvement over the Logistic Regression baseline means fewer customers slip through the cracks.

In [ ]:
# Display ROC and Precision-Recall curvesfig, axes = plt.subplots(1, 3, figsize=(20, 6))for ax, fname, title in zip(axes,    ['18_roc_curves.png', '19_precision_recall.png', '20_confusion_matrix.png'],    ['ROC Curves — 3-Model Comparison', 'Precision-Recall Curve', 'Confusion Matrix (Tuned)']):    img = mpimg.imread(FIG_DIR / fname)    ax.imshow(img)    ax.axis('off')    ax.set_title(title, fontsize=13, fontweight='bold')plt.tight_layout()plt.show()

---

## 5. What Is Each Customer Worth? — Customer Lifetime ValueCLV was estimated using the formula:$$\text{CLV} = \text{MonthlyCharges} \times \text{Expected Tenure} \times (1 - \text{Churn Probability})$$Where expected tenure comes from a Random Forest Regressor and churn probability from Phase 4's best model.

In [ ]:
# CLV distribution by clusterfig, axes = plt.subplots(1, 2, figsize=(16, 6))img1 = mpimg.imread(FIG_DIR / '26_clv_by_cluster.png')axes[0].imshow(img1)axes[0].axis('off')axes[0].set_title('CLV Distribution by Customer Segment', fontsize=13, fontweight='bold')img2 = mpimg.imread(FIG_DIR / '28_clv_churn_heatmap.png')axes[1].imshow(img2)axes[1].axis('off')axes[1].set_title('CLV Tier vs. Churn Risk', fontsize=13, fontweight='bold')plt.tight_layout()plt.show()

In [ ]:
# CLV summary statisticsclv_summary = clv.groupby('Cluster').agg(    count=('CLV', 'count'),    median_clv=('CLV', 'median'),    mean_clv=('CLV', 'mean'),    avg_churn_prob=('churn_proba', 'mean')).round(2)clv_summary.columns = ['Customers', 'Median CLV ($)', 'Mean CLV ($)', 'Avg Churn Probability']print("=" * 70)print("CLV BY SEGMENT")print("=" * 70)display(clv_summary)print(f"\nTotal Portfolio CLV: ${clv['CLV'].sum():,.0f}")print(f"Average CLV per Customer: ${clv['CLV'].mean():,.2f}")print(f"CLV Range: ${clv['CLV'].min():,.2f} — ${clv['CLV'].max():,.2f}")print(f"\nPareto Effect: Top 20% of customers generate 54.2% of total CLV")

In [ ]:
# Pareto chartfig, ax = plt.subplots(1, 1, figsize=(10, 6))img = mpimg.imread(FIG_DIR / '30_pareto_clv.png')ax.imshow(img)ax.axis('off')ax.set_title('Pareto Analysis — CLV Concentration', fontsize=14, fontweight='bold')plt.tight_layout()plt.show()

### Key Takeaway — CLV> **The highest-CLV cluster generates 5.2× the revenue potential of the lowest.** The "At-Risk" segment paradoxically has the highest CLV because preventing their churn unlocks the most revenue. Meanwhile, the Pareto effect is strong: the top 20% of customers represent 54.2% of total lifetime value.---

## 6. Data-Driven Business Recommendations| # | Recommendation | Supporting Evidence | Target Segment | Expected Impact ||---|---|---|---|---|| 1 | **Launch contract upgrade incentives** | Contract type is the #1 churn driver (SHAP = 0.048); M-to-M churn rate = 42% vs. 2-year = 3% | 🔴 At-Risk Churners | 10% conversion → saves ~157 customers, ~$515K CLV || 2 | **Implement 90-day onboarding program** | Tenure is the #2 churn driver; early-life churn vulnerability confirmed in KDE plots | 🟡 New Potentials | Reduces first-year attrition || 3 | **Audit fiber optic service quality** | Fiber customers churn at elevated rates despite premium pricing (SHAP = 0.037) | All fiber subscribers | Resolves price-value mismatch || 4 | **Cross-sell services to increase stickiness** | Each additional service reduces churn risk; `services_count` is a direct predictor | 🟡 New Potentials | Higher engagement → lower churn || 5 | **Deploy real-time churn scoring** | Model catches 82% of churners; can flag at-risk accounts for proactive outreach | All customers | Early intervention before churn occurs |---

## 7. Quantified Resume Bullets```• Built end-to-end churn prediction pipeline using Random Forest, achieving AUC-ROC of 0.84  and F1-score of 0.63; applied SHAP explainability to surface top 5 churn drivers.• Segmented 7,043 customers into 3 behavioral clusters using K-Means (silhouette = 0.42);  identified at-risk segment representing 53% of customers but 84.2% of predicted churn.• Estimated Customer Lifetime Value (CLV) by combining Kaplan-Meier survival analysis  with churn probability scores; highest-CLV cluster showed 5.2× revenue potential  vs. lowest.• Improved churner recall from 69.8% (Logistic Regression baseline) to 81.6% through  threshold tuning and class imbalance handling (SMOTE) — a 16.9% improvement.```---

## 8. End-to-End Pipeline Architecture```┌──────────────┐    ┌──────────────────┐    ┌──────────────────┐│  Raw Data    │───▸│  Preprocessing   │───▸│  Segmentation    ││  (7,043 × 21)│    │  + 5 Engineered  │    │  K-Means (K=3)   ││              │    │    Features       │    │  Silhouette=0.42 │└──────────────┘    └──────────────────┘    └────────┬─────────┘                                                     │                           ┌─────────────────────────┘                           ▼                    ┌──────────────────┐    ┌──────────────────┐                    │  Churn Prediction│───▸│  CLV Estimation  │                    │  Random Forest   │    │  Survival + ML   │                    │  AUC=0.84        │    │  Range: $11-$7.8K│                    └──────────────────┘    └────────┬─────────┘                                                     │                           ┌─────────────────────────┘                           ▼                    ┌──────────────────────────────────┐                    │  Business Recommendations        │                    │  5 data-driven retention actions  │                    └──────────────────────────────────┘```---## 📁 Artifacts Produced| Category | Count | Details ||---|---|---|| **Charts** | 30 | Saved in `reports/figures/` at 300 DPI || **Models** | 2 | Random Forest (churn), K-Means (segmentation) || **Datasets** | 8 | Raw, preprocessed, segmented, CLV-scored, train/test splits || **Notebooks** | 6 | One per phase, fully executable top-to-bottom |---*Analysis completed — Customer Segmentation & Retention Analysis Project*